In [1]:
!pip install transformers datasets seqeval

  Using cached huggingface_hub-1.9.0-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached huggingface_hub-1.9.0-py3-none-any.whl (637 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.6
    Uninstalling datasets-2.14.6:
      Successfully uninstalled datasets-2.14.6


In [2]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer

In [4]:
data = [
    (["John", "works", "at", "Google"], ["NOUN", "VERB", "ADP", "PROPN"]),
    (["She", "is", "reading", "a", "book"], ["PRON", "AUX", "VERB", "DET", "NOUN"]),
    (["They", "live", "in", "California"], ["PRON", "VERB", "ADP", "PROPN"]),
    (["He", "plays", "cricket"], ["PRON", "VERB", "NOUN"]),
]

In [5]:
unique_labels = list(set(label for sent in data for label in sent[1]))

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def encode_data(data):

    encodings = []
    labels = []

    for tokens, tags in data:

        encoded = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",   # ⭐ IMPORTANT
            max_length=10           # small fixed length
        )

        word_ids = encoded.word_ids()
        label_ids = []

        prev_word = None

        for word_id in word_ids:

            if word_id is None:
                label_ids.append(-100)

            elif word_id != prev_word:
                label_ids.append(label2id[tags[word_id]])

            else:
                label_ids.append(-100)

            prev_word = word_id

        labels.append(label_ids)
        encodings.append(encoded)

    return encodings, labels

In [16]:
encodings, labels = encode_data(data)

In [17]:
import torch

class CustomDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [18]:
train_dataset = CustomDataset(encodings, labels)

In [19]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3
)

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [22]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=1.7277522087097168, metrics={'train_runtime': 13.2448, 'train_samples_per_second': 0.906, 'train_steps_per_second': 0.453, 'total_flos': 61244195760.0, 'train_loss': 1.7277522087097168, 'epoch': 3.0})

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset   # reuse same data
)

In [27]:

results = trainer.evaluate()
print(results)

{'eval_loss': 1.4802175760269165, 'eval_model_preparation_time': 0.0047, 'eval_runtime': 0.0612, 'eval_samples_per_second': 65.402, 'eval_steps_per_second': 16.351}


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
model.to(device)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [32]:
sentence = ["John", "works", "at", "Google"]

inputs = tokenizer(
    sentence,
    is_split_into_words=True,
    return_tensors="pt",
    padding="max_length",
    max_length=10
)

inputs = {key: val.to(device) for key, val in inputs.items()}

outputs = model(**inputs)

preds = outputs.logits.argmax(dim=2)

predicted_tags = [id2label[p.item()] for p in preds[0]]

for word, tag in zip(sentence, predicted_tags[:len(sentence)]):
    print(f"{word} → {tag}")

John → AUX
works → PRON
at → VERB
Google → ADP


## Report: POS Tagging vs Chunking

### Overview

In this task, a transformer-based model (BERT) was used to perform token classification. The goal was to assign labels to each word in a sentence based on its role in the language.

---

### POS Tagging vs Chunking

Part-of-Speech (POS) tagging focuses on identifying the grammatical role of each word in a sentence, such as noun, verb, adjective, etc. It works at the word level and helps in understanding the basic structure of a sentence.

Chunking, on the other hand, groups words into meaningful phrases like noun phrases or verb phrases. It provides higher-level structure by identifying how words are related within a sentence.

In simple terms:
- POS tagging → word-level classification  
- Chunking → phrase-level grouping  

---

### Challenges Faced

- Handling tokenization was challenging because BERT splits words into subwords.
- Aligning original labels with tokenized inputs required careful implementation.
- Managing padding and ensuring all sequences have equal length was necessary for training.

---

### Observations

- The model was able to assign labels to tokens effectively even with a small dataset.
- Proper preprocessing and label alignment significantly impact model performance.
- Transformer models like BERT are powerful for sequence labeling tasks.

---

### Conclusion

This task demonstrates the effectiveness of transformer-based models for token classification tasks. By fine-tuning BERT, we can accurately perform POS tagging and similar NLP tasks. Understanding preprocessing and token-label alignment is essential for achieving good results.